In [0]:
%sql
CREATE OR REPLACE TEMPORARY FUNCTION tcp_ping(
    host STRING,
    port INT,
    timeout_seconds DOUBLE
)
RETURNS STRING
LANGUAGE PYTHON
AS $$
import socket
import time

started = time.perf_counter()

try:
    with socket.create_connection(
        (host, port),
        timeout=timeout_seconds
    ):
        elapsed_ms = round((time.perf_counter() - started) * 1000, 2)
        return f"CONNECTED in {elapsed_ms} ms"

except Exception as error:
    elapsed_ms = round((time.perf_counter() - started) * 1000, 2)
    return f"FAILED after {elapsed_ms} ms: {type(error).__name__}: {error}"
$$;

In [0]:
from pyspark.sql import functions as F

number_of_tests = 8

result = (
    spark.range(number_of_tests)
    .repartition(number_of_tests)
    .select(
        F.col("id").alias("test_id"),
        F.spark_partition_id().alias("partition_id"),
        F.expr(
            "tcp_ping('www.google.com', 443, 3.0)"
        ).alias("internet_test"),
    )
)

display(result)